# Validation speaker inference — Qwen2.5-VL-3B

This is the next step after transcript preparation and prompt creation. It runs the pretrained model on three annotated images together, using existing OCR text. It does not change OCR, groups, labels, or prompts.

Outputs: `raw_responses.jsonl`, `predictions.jsonl`, `prediction_audit.jsonl`, `evaluation.jsonl`, `summary.json`, and `run_metadata.json` in a dedicated run folder. Predictions checkpoint after each sequence and can resume only with matching inputs and settings.

The baseline uses greedy decoding without constrained decoding. UNKNOWN and NARRATION are fixed classes; character names are matched once across all three pages during evaluation. This is accepted-group speaker accuracy, not the official end-to-end transcript score.

Cells are left unexecuted. First rerun `build_speaker_prompt_Vl.ipynb` if transcript preparation has changed.


## Existing environment and model

Select **AppliedTaskByte**, your `.venvAppliedTask` kernel. Required inference packages are already installed. The notebook records their versions for each run.

Load the model directly from `C:/MODELS/HF/Qwen2.5-VL-3B-Instruct`. All model and processor loads use `local_files_only=True`; no weights are downloaded.

References: [Qwen model usage](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), [bitsandbytes installation](https://huggingface.co/docs/bitsandbytes/installation).


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import random
import re
import time

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'TaskData').is_dir() or (p / 'manga_notebook_codebase/TaskData').is_dir())
if not (ROOT / 'TaskData').is_dir():
    ROOT = ROOT / 'manga_notebook_codebase'
ROOT = ROOT.resolve()
PROMPT_DIR = ROOT / 'outputs/speaker_prompts/val'
PREP_DIR = ROOT / 'outputs/transcript_prep_vl/val'
RUN_NAME = 'qwen25vl3b_speaker_lines_baseline_v1'
OUTPUT_DIR = ROOT / 'outputs/vl_inference/val' / RUN_NAME
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MODEL_PATH = Path(r'C:\MODELS\HF\Qwen2.5-VL-3B-Instruct')  # Existing local weights.
ADAPTER_PATH = None  # Baseline first; then set the new ordered_speaker_lora_v1 adapter/checkpoint.
USE_4BIT = True  # Reduce GPU memory use on the 8 GB GPU.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28  # Budget per image, applied to all three pages.
MAX_NEW_TOKENS = 2048
SEED = 42  # Record the same seed for every run.
LIMIT = None  # Set 1 for a smoke run, then None to resume the same run on all sequences.

import sys
import importlib
sys.path.insert(0, str(ROOT / 'src'))
import vl_prompts
importlib.reload(vl_prompts)  # Pick up edits even in an existing kernel.
from vl_prompts import (OUTPUT_FORMAT, ordered_groups, build_speaker_prompt,
                        build_speaker_target, parse_speaker_output, permute_speaker_labels)


## 1. Check inputs before loading the model

Fail early on stale prompts, wrong split, changed images, or mismatched group IDs. Target speakers do not enter the inference messages. Input and image hashes are part of the run identity.


In [ ]:
def file_hash(path):
    """Identify a file by its contents without loading the whole file into memory."""
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()
            if line.strip()]


def unique_records(records):
    """Index records by sequence ID and reject duplicate sequences."""
    result = {r['sequence_id']: r for r in records}
    if len(result) != len(records):
        raise ValueError('Duplicate sequence IDs')
    return result


def save_json(path, value):
    """Replace the file only after the new contents have been written."""
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
                         encoding='utf-8')
    os.replace(temporary, path)


def save_jsonl(path, records):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(''.join(json.dumps(r, ensure_ascii=False, allow_nan=False) + '\n'
                                  for r in records), encoding='utf-8')
    os.replace(temporary, path)


# Check that prompts still correspond to the latest preparation outputs.
prompt_summary = json.loads((PROMPT_DIR / 'summary.json').read_text(encoding='utf-8'))
if prompt_summary.get('output_format') != OUTPUT_FORMAT:
    raise ValueError('Rerun build_speaker_prompt_Vl.ipynb for ordered speaker outputs.')
if file_hash(ROOT / 'src/vl_prompts.py') != prompt_summary['shared_prompt_sha256']:
    raise ValueError('Shared prompt changed; regenerate prompts.')
for name, expected in prompt_summary['output_sha256'].items():
    if file_hash(PROMPT_DIR / name) != expected:
        raise ValueError(f'{name} changed after prompt creation; rerun the prompt notebook.')
for name, key in [('inputs.jsonl', 'inputs'), ('targets.jsonl', 'targets')]:
    if file_hash(PREP_DIR / name) != prompt_summary['source_sha256'][key]:
        raise ValueError('Transcript preparation changed: rerun build_speaker_prompt_Vl.ipynb.')
split_path = ROOT / 'outputs/split.json'
if file_hash(split_path) != prompt_summary['source_sha256']['split']:
    raise ValueError('The saved split changed after prompt creation.')
prompts = read_jsonl(PROMPT_DIR / 'prompts.jsonl')
prepared = unique_records(read_jsonl(PREP_DIR / 'inputs.jsonl'))
by_id = unique_records(prompts)
val_ids = set(json.loads(split_path.read_text(encoding='utf-8'))['val'])
if set(by_id) != set(prepared) or not set(by_id).issubset(val_ids):
    raise ValueError('Prompt/prepared IDs differ or include a non-validation sequence.')
# Keep expected IDs for format checking; verify the three image attachments.
expected_ids, image_hashes = {}, {}
for record in prompts:
    sid = record['sequence_id']
    pages = prepared[sid]['pages']
    if len(pages) != 3 or [p['page_index'] for p in pages] != [0, 1, 2]:
        raise ValueError(f'{sid}: invalid page order')
    expected_images = [str((PREP_DIR / p['image']).resolve()) for p in pages]
    if record['images'] != expected_images:
        raise ValueError(f'{sid}: image attachment order differs from preparation')
    requested = ordered_groups(pages)
    if record['requested_groups'] != requested or record['prompt'] != build_speaker_prompt(requested):
        raise ValueError(f'{sid}: ordered groups or prompt differ from preparation')
    ids = [g['group_id'] for g in requested]
    if len(ids) != len(set(ids)):
        raise ValueError(f'{sid}: duplicate group IDs')
    expected_ids[sid] = ids
    for image in record['images']:
        path = Path(image).resolve()
        if not path.is_relative_to(PREP_DIR.resolve()) or not path.is_file():
            raise ValueError(f'Missing or unsafe image: {path}')
        relative = str(path.relative_to(PREP_DIR.resolve()))
        expected = prompt_summary['annotated_image_sha256'][relative]
        actual = file_hash(path)
        if actual != expected:
            raise ValueError(f'Annotated image changed: {path}')
        image_hashes[relative] = actual
if LIMIT is not None and (type(LIMIT) is not int or LIMIT < 1):
    raise ValueError('LIMIT must be a positive integer or None')
selected = prompts[:LIMIT] if LIMIT is not None else prompts
print(f'Ready: {len(selected)} sequences, {sum(len(expected_ids[r["sequence_id"]]) for r in selected)} groups')


## 2. Load the local model

NF4 quantization and the image budget are explicit baseline choices for an 8 GB GPU. They can affect accuracy. Changing either requires a new `RUN_NAME`; there is no silent fallback to a different resolution or model.

Greedy decoding and fixed seeds reduce variation, but exact equality across different hardware or library versions is not guaranteed. The run records installed package versions, GPU, model file hashes, and notebook code.


In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

if not torch.cuda.is_available():
    raise RuntimeError('Select the CUDA-enabled .venvAppliedTask kernel; this baseline requires a GPU.')
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
# Confirm that the local folder contains every weight shard.
model_folder = MODEL_PATH.resolve()
if not model_folder.is_dir():
    raise FileNotFoundError(f'Model folder not found: {model_folder}')
index_path = model_folder / 'model.safetensors.index.json'
index = json.loads(index_path.read_text(encoding='utf-8'))
for shard in set(index['weight_map'].values()):
    if not (model_folder / shard).is_file():
        raise FileNotFoundError(f'Missing model weight shard: {shard}')

# Hash local files so a run can identify exactly which weights it used.
model_hashes = {p.name: file_hash(p) for p in sorted(model_folder.iterdir()) if p.is_file()}
# The processor turns images and prompt text into model inputs.
processor = AutoProcessor.from_pretrained(
    model_folder, local_files_only=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
# SDPA works without installing FlashAttention.
load_options = dict(dtype=compute_dtype, device_map={'': 0},
                    attn_implementation='sdpa', local_files_only=True)
# NF4 stores weights in 4 bits; computations use the GPU-supported dtype.
if USE_4BIT:
    load_options['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype)
# Load weights once and switch off training behavior.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_folder, **load_options)
# Optionally evaluate a trained LoRA adapter with the same base and input settings.
adapter_hashes = None
if ADAPTER_PATH is not None:
    from peft import PeftModel
    adapter_folder = Path(ADAPTER_PATH).resolve()
    model = PeftModel.from_pretrained(model, adapter_folder, local_files_only=True)
    adapter_hashes = {path.name: file_hash(path) for path in adapter_folder.iterdir()
                      if path.is_file() and path.suffix in {'.json', '.safetensors'}}
model.eval()
# Record the settings, runtime, and source hashes before making predictions.
notebook = ROOT / 'notebooks/validation_VL.ipynb'
code_sources = [c['source'] for c in json.loads(notebook.read_text(encoding='utf-8'))['cells']
                if c['cell_type'] == 'code']
packages = {name: importlib.metadata.version(name)
            for name in ['torch', 'torchvision', 'transformers', 'accelerate',
                         'qwen-vl-utils', 'bitsandbytes', 'Pillow', 'huggingface-hub']}
run_config = dict(output_format=OUTPUT_FORMAT, model_id=MODEL_ID, model_path=str(model_folder), model_sha256=model_hashes,
                  adapter_path=str(Path(ADAPTER_PATH).resolve()) if ADAPTER_PATH is not None else None,
                  adapter_sha256=adapter_hashes,
                  use_4bit=USE_4BIT, compute_dtype=str(compute_dtype),
                  min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, max_new_tokens=MAX_NEW_TOKENS,
                  seed=SEED, do_sample=False, num_beams=1, constrained_decoding=False,
                  prompt_sha256=file_hash(PROMPT_DIR / 'prompts.jsonl'),
                  prepared_inputs_sha256=file_hash(PREP_DIR / 'inputs.jsonl'),
                  image_sha256=image_hashes, packages=packages,
                  notebook_code_sha256=hashlib.sha256(json.dumps(code_sources).encode()).hexdigest(),
                  gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,
                  python=platform.python_version(), platform=platform.platform())
# LIMIT is intentionally omitted: a smoke run can resume on the remaining sequences.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata_path = OUTPUT_DIR / 'run_metadata.json'
if metadata_path.exists():
    if json.loads(metadata_path.read_text(encoding='utf-8')) != run_config:
        if (OUTPUT_DIR / 'raw_responses.jsonl').exists():
            raise ValueError('Run configuration changed. Choose a new RUN_NAME.')
        # A failed run with no predictions can safely refresh its metadata.
        save_json(metadata_path, run_config)
else:
    if (OUTPUT_DIR / 'raw_responses.jsonl').exists():
        raise ValueError('Existing responses have no metadata; choose a new RUN_NAME.')
    save_json(metadata_path, run_config)
print('Loaded', MODEL_ID, 'on', run_config['gpu'])


## 3. Infer and checkpoint each sequence

Image paths are passed as image entries, not pasted into the text prompt. Three pages are supplied together. No target speakers are loaded here. Raw text is saved before parsing, including malformed responses and generation-limit information. There are no label-guided retries or repairs.


In [ ]:
from PIL import Image


def load_image_content(image_paths):
    # Open files ourselves. Qwen receives pixels, so it never parses a Windows URI.
    content = []
    for image_path in image_paths:
        with Image.open(Path(image_path)) as source:
            image = source.convert('RGB')
        content.append(dict(type='image', image=image,
                            min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS))
    return content


def infer_sequence(record):
    # Attach three image objects in page order, followed by the OCR prompt.
    content = load_image_content(record['images'])
    content.append(dict(type='text', text=record['prompt']))
    messages = [dict(role='user', content=content)]
    # Apply Qwen's chat format and prepare the image tensors.
    rendered_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    # Resize images using the fixed budget, then convert them to model tensors.
    images, videos = process_vision_info(messages)
    batch = processor(text=[rendered_prompt], images=images, videos=videos,
                      padding=True, return_tensors='pt').to('cuda')
    # Image pixels are now in tensors; release the temporary PIL images.
    for item in content:
        if item['type'] == 'image':
            item['image'].close()
    for image in images:
        image.close()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    # Generate one greedy response; reference speakers are never supplied.
    with torch.inference_mode():
        generated = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS,
                                   do_sample=False, num_beams=1, use_cache=True)
    torch.cuda.synchronize()
    # Remove the input tokens so only the generated answer is decoded.
    output_ids = generated[0, batch['input_ids'].shape[1]:]
    text = processor.decode(output_ids, skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)
    result = dict(sequence_id=record['sequence_id'], raw_response=text,
                  input_tokens=int(batch['input_ids'].shape[1]),
                  output_tokens=int(output_ids.numel()),
                  hit_token_limit=bool(output_ids.numel() >= MAX_NEW_TOKENS),
                  elapsed_seconds=time.perf_counter() - start,
                  peak_gpu_allocated_gb=torch.cuda.max_memory_allocated() / 1024**3,
                  image_grid_thw=batch['image_grid_thw'].tolist())
    del batch, generated, output_ids
    return result


# Save after each sequence so an interrupted run can resume.
raw_path = OUTPUT_DIR / 'raw_responses.jsonl'
# Refresh code provenance after a fix when the failed run saved no predictions.
current_sources = [c['source'] for c in json.loads(notebook.read_text(encoding='utf-8'))['cells']
                   if c['cell_type'] == 'code']
current_code_hash = hashlib.sha256(json.dumps(current_sources).encode()).hexdigest()
if current_code_hash != run_config['notebook_code_sha256']:
    if raw_path.exists():
        raise ValueError('Notebook code changed after predictions. Choose a new RUN_NAME.')
    run_config['notebook_code_sha256'] = current_code_hash
    save_json(metadata_path, run_config)
completed = unique_records(read_jsonl(raw_path)) if raw_path.exists() else {}
if not set(completed).issubset(set(by_id)):
    raise ValueError('Saved responses contain unexpected sequences.')
for record in selected:
    sid = record['sequence_id']
    if sid in completed:
        print('Reusing', sid)
        continue
    completed[sid] = infer_sequence(record)
    save_jsonl(raw_path, [completed[r['sequence_id']] for r in prompts
                         if r['sequence_id'] in completed])
    print(sid, completed[sid]['output_tokens'], 'tokens',
          round(completed[sid]['elapsed_seconds'], 1), 'seconds', flush=True)


## 4. Validate the response format

Require exactly one valid label per requested group, in saved order. Reject the entire response on wrong counts or invalid labels: accepting a shortened list could shift every later assignment. JSON below is only the saved evaluation format, not model output.


In [ ]:
def validate_response(text, requested_groups):
    # Keep strict format errors separate from speaker accuracy.
    audit = dict(valid_format=False, expected_labels=len(requested_groups),
                 returned_labels=len([s for s in text.splitlines() if s.strip()]),
                 missing_ids=[], complete_valid_mapping=False, parse_error=None)
    try:
        speakers = parse_speaker_output(text, requested_groups)
    except (ValueError, TypeError) as error:
        audit.update(parse_error=str(error),
                     missing_ids=[g['group_id'] for g in requested_groups])
        return {}, audit
    audit.update(valid_format=True, complete_valid_mapping=True)
    return speakers, audit


predictions, audits = [], []
for record in prompts:
    sid = record['sequence_id']
    if sid not in completed:
        continue
    # Reattach IDs after parsing, using exactly the order sent to the model.
    speakers, audit = validate_response(completed[sid]['raw_response'], record['requested_groups'])
    predictions.append(dict(sequence_id=sid, speakers=speakers))
    audits.append(dict(sequence_id=sid, **audit))
save_jsonl(OUTPUT_DIR / 'predictions.jsonl', predictions)
save_jsonl(OUTPUT_DIR / 'prediction_audit.jsonl', audits)
print('Complete valid responses:', sum(a['complete_valid_mapping'] for a in audits), '/', len(audits))


## 5. Score accepted targets

Character numbering is arbitrary. Reuse the official scorer's one-to-one Hungarian matching helper with unit weight per accepted OCR group, once across the full sequence. UNKNOWN and NARRATION stay fixed and never enter character matching. This checks identity consistency across pages; it is not direct grounding to character bounding boxes.

Score all accepted targets, including missing predictions as incorrect. Groups without accepted targets remain unscored. Report labelled coverage so this subset metric is not mistaken for performance on every group. Evaluation is post-inference and cannot change predictions.


In [ ]:
# Reuse the official one-to-one character matching algorithm.
scorer_path = ROOT / 'TaskData/score.py'
spec = importlib.util.spec_from_file_location('official_manga_scorer', scorer_path)
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)


def score_sequence(predicted, gold):
    # Match character IDs across the full sequence, keeping special classes fixed.
    fixed = {'UNKNOWN', 'NARRATION'}
    pairs = [(target, predicted[gid], 1.0) for gid, target in gold.items()
             if gid in predicted and target not in fixed and predicted[gid] not in fixed]
    mapping = scorer.best_mapping(pairs)
    mapping.update(UNKNOWN='UNKNOWN', NARRATION='NARRATION')
    # Missing predictions count as errors instead of disappearing from the score.
    details = []
    for gid, target in gold.items():
        prediction = predicted.get(gid)
        mapped = mapping.get(prediction)
        details.append(dict(group_id=gid, target=target, prediction=prediction,
                            mapped_prediction=mapped, correct=mapped == target))
    correct = sum(d['correct'] for d in details)
    return dict(label_mapping=mapping, labelled_groups=len(gold), correct=correct,
                accuracy=correct / len(gold) if gold else None, groups=details)


# Load answers only now, after inference has finished.
references = unique_records(read_jsonl(PROMPT_DIR / 'scoring_targets.jsonl'))
if set(references) != set(by_id):
    raise ValueError('Scoring reference sequence IDs differ from prompts.')
evaluations = []
for prediction in predictions:
    sid = prediction['sequence_id']
    gold = references[sid]['speakers']
    if not set(gold).issubset(set(expected_ids[sid])):
        raise ValueError(f'{sid}: targets contain unexpected group IDs')
    evaluations.append(dict(sequence_id=sid, **score_sequence(prediction['speakers'], gold)))
save_jsonl(OUTPUT_DIR / 'evaluation.jsonl', evaluations)
labelled = sum(e['labelled_groups'] for e in evaluations)
correct = sum(e['correct'] for e in evaluations)
all_groups = sum(len(expected_ids[e['sequence_id']]) for e in evaluations)
class_counts = Counter(d['target'] if d['target'] in {'UNKNOWN', 'NARRATION'} else 'character'
                       for e in evaluations for d in e['groups'])
class_correct = Counter(d['target'] if d['target'] in {'UNKNOWN', 'NARRATION'} else 'character'
                        for e in evaluations for d in e['groups'] if d['correct'])
summary = dict(metric='Accepted-group speaker accuracy, sequence-wide character permutation',
               completed_sequences=len(evaluations), available_sequences=len(prompts),
               input_groups=all_groups, labelled_groups=labelled,
               labelled_coverage=labelled / all_groups if all_groups else None,
               correct=correct, speaker_accuracy=correct / labelled if labelled else None,
               valid_format_responses=sum(a['valid_format'] for a in audits),
               complete_valid_responses=sum(a['complete_valid_mapping'] for a in audits),
               missing_predictions=sum(len(a['missing_ids']) for a in audits),
               format_errors=sum(a['parse_error'] is not None for a in audits),
               token_limit_responses=sum(r['hit_token_limit'] for r in completed.values()),
               class_accuracy={label: dict(count=count, correct=class_correct[label],
                                          accuracy=class_correct[label] / count)
                               for label, count in class_counts.items()},
               scoring_targets_sha256=file_hash(PROMPT_DIR / 'scoring_targets.jsonl'),
               scorer_sha256=file_hash(scorer_path),
               output_sha256={name: file_hash(OUTPUT_DIR / name) for name in
                              ['raw_responses.jsonl', 'predictions.jsonl',
                               'prediction_audit.jsonl', 'evaluation.jsonl']})
save_json(OUTPUT_DIR / 'summary.json', summary)
print(json.dumps(summary, indent=2))


## 6. Inspect a response and its errors

Review predictions without modifying them. If the smoke run works, change `LIMIT` from 1 to None and rerun. Change the run name when changing model, prompts, resolution, quantization, or decoding settings. The baseline does not improve grouping or tune alignment to its own predictions.


In [ ]:
if evaluations:
    sid = evaluations[0]['sequence_id']
    print('Sequence:', sid)
    print('Raw response:', completed[sid]['raw_response'])
    print('Label mapping:', evaluations[0]['label_mapping'])
    print('Incorrect or missing:', json.dumps(
        [g for g in evaluations[0]['groups'] if not g['correct']], indent=2))
